# TakeMeter — training notebook

This notebook trains your classifier. It runs **on your own machine**, from
inside your project folder — no accounts, no upload, nothing to connect.

**Before you run anything:**

1. **Open it from your repo**, not from a copy somewhere else. Everything it
   writes lands next to your code so you can commit it normally.
2. **Activate your virtual environment first.** In VS Code, that's the kernel
   picker in the top right — choose the `.venv` inside this project. Getting
   this wrong is the single most common way the first cell fails.

How long it takes depends on what you're running on. Section 1 tells you, and
the first cell of section 4 prints a time as it goes. On a laptop CPU expect a
few minutes per seed; on a GPU, well under one.

---

### What's here

| Section | When | What it does |
|---|---|---|
| **1. Setup** | unit 5 | Checks what you're running on and where |
| **2. Your labels** | unit 5 | Your label map, and reading `labels.csv` |
| **3. Split** | unit 5 | 70/15/15, with the seed as an editable constant |
| **4. Train** | unit 5 | The training run |
| **5. Results** | unit 5 | Writes `results.json` and `test_split.csv` |
| **6. Three seeds** | unit 6 | The same thing three times, on three splits |
| **7. Confusion matrix** | unit 6 | Which labels get mixed up, as a table |

## 1. Setup

Two cells. Run them once at the start of a session.

The first tells you what hardware you're training on and records it, because
**that is part of your result**. The second confirms the notebook can see your
project files.

In [20]:
# Everything this notebook needs is already in requirements.txt — you installed
# it when you set the project up. This cell just says what you're running on.

import torch


def pick_device():
    """
    Where training will run. Fastest first.

    `cuda`  an NVIDIA GPU.
    `mps`   Apple Silicon's GPU — M1 and later. Much faster than the CPU path.
    `cpu`   everything else. Slower, and it works: this dataset is small enough
            that a full three-seed run is minutes, not hours.
    """
    if torch.cuda.is_available():
        return "cuda", torch.cuda.get_device_name(0)
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps", "Apple Silicon GPU (MPS)"
    return "cpu", "CPU"


DEVICE, DEVICE_NAME = pick_device()

print(f"device: {DEVICE}  ({DEVICE_NAME})")
print(f"torch:  {torch.__version__}")
print()
print("⚠️  WRITE THIS DOWN. Your results depend on it.")
print()
print("Two people running this notebook on the same data with the same seed can")
print("get different numbers, because different hardware does the arithmetic in")
print("a slightly different order. That is not a bug and it is not one of you")
print("being wrong — it is a real property of the thing you are building, and")
print("unit 6 asks you to report it alongside your three seeds.")

device: mps  (Apple Silicon GPU (MPS))
torch:  2.14.1

⚠️  WRITE THIS DOWN. Your results depend on it.

Two people running this notebook on the same data with the same seed can
get different numbers, because different hardware does the arithmetic in
a slightly different order. That is not a bug and it is not one of you
being wrong — it is a real property of the thing you are building, and
unit 6 asks you to report it alongside your three seeds.


### Where you are

Nothing to connect and nothing to clone — this notebook runs inside your own
repo, so `labels.csv` is simply *there* and anything written lands beside it.

**Fill in the two lines.** They only label your results; nothing here talks to
GitHub. You commit what this notebook writes the same way you commit anything
else.

In [21]:
# ── FILL THIS IN ─────────────────────────────────────────────────────────────

NAME  = "Smit Chandi"          # only used to label your results
EMAIL = "smitchandi@gmail.com"    # same — nothing here talks to GitHub

# ─────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path

# This notebook runs from inside your own repo, so there is nothing to clone
# and nothing to authenticate. Everything it writes lands next to your code and
# you commit it the same way you commit anything else.
REPO_DIR = Path.cwd()

expected = ["labels.csv", "criteria.md", "baseline.py"]
missing = [f for f in expected if not (REPO_DIR / f).exists()]

if missing and (REPO_DIR / "takemeter.ipynb").exists():
    print(f"Working in: {REPO_DIR}")
    print(f"[note] not found yet: {', '.join(missing)}")
    print("       labels.csv is Milestone 4 — that's expected before then.")
elif missing:
    print("This notebook expects to be run from inside your project folder.")
    print(f"It is currently running in: {REPO_DIR}")
    print("Open the notebook from the repo you cloned, not from a copy.")
else:
    print(f"Working in: {REPO_DIR}")
    print(f"Results will be labelled: {NAME} <{EMAIL}>")
    for f in sorted(os.listdir(REPO_DIR)):
        if not f.startswith("."):
            print(f"  {f}")

Working in: /Users/smitchandi/Documents/Coding/ai201-p3_Model-Finetuning
Results will be labelled: Smit Chandi <smitchandi@gmail.com>
  Helpful_Data
  README.md
  RUNNING.md
  agreement.py
  agreement_results.json
  ai201-p3
  baseline.py
  baseline_results.json
  criteria.md
  data
  label_definitions.txt
  labels.csv
  labels_template.csv
  my_staff_labels.csv
  requirements.txt
  results.json
  results_three_seeds_before.json
  takemeter.ipynb
  takemeter_out
  test.py
  test_split.csv


## 2. Your labels

Two things to fill in. Everything after this is automatic.

**`LABELS` must match your CSV exactly** — including capitals and spaces. A
label in your file that isn't in this list is the single most common reason
this notebook fails.

In [22]:
# ── FILL THIS IN ─────────────────────────────────────────────────────────────

# Your labels, exactly as they appear in the `label` column of your CSV.
LABELS = ["rule_question", "experience_request", "risk_check", "miscellaneous"]

# The model you're fine-tuning. Leave this unless you have a reason.
BASE_MODEL = "distilbert-base-uncased"

# ── Training settings ────────────────────────────────────────────────────────
# These are the defaults. Hyperparameters are vocabulary this unit, not
# decisions — if you change one, write down what and why in your README.

EPOCHS = 10
LEARNING_RATE = 2e-5
BATCH_SIZE = 16

# How many tokens of each post the model reads. Anything longer is truncated,
# anything shorter is padded up to it — so this is really "how much work per
# post", and it costs time and memory whether your posts are that long or not.
#
# 128 covers almost every post in a corpus like this. Raising it to 256 roughly
# doubles both the time and the memory for text most people don't have.
MAX_LENGTH = 128

# ── The seed ─────────────────────────────────────────────────────────────────
# This fixes which posts land in which split. Change it and you get a different
# split and different numbers.
#
# ⚠️ NEXT UNIT YOU CHANGE THIS THREE TIMES. That's not busywork — it's how you
# find out whether your number is real or whether you got a friendly draw.

SEED = 42

print(f"{len(LABELS)} labels: {', '.join(LABELS)}")
print(f"seed {SEED}, {EPOCHS} epochs, lr {LEARNING_RATE}, batch {BATCH_SIZE}")

4 labels: rule_question, experience_request, risk_check, miscellaneous
seed 42, 10 epochs, lr 2e-05, batch 16


### Your data

`labels.csv` came down with the clone, so there's nothing to upload. If you've
just added rows on your laptop, commit and push them there first, then re-run
the connect cell.

Don't split the file yourself — the next section does that, and doing it twice
is how examples leak between training and test.

No data of your own yet? Point `CSV` at `data/practice_labels.csv`, the 60-post
set used in class.

In [23]:
import pandas as pd

CSV = "labels.csv"              # "data/practice_labels.csv" to practise

df = pd.read_csv(CSV)

print(f"Loaded {CSV}: {len(df)} rows")
print(f"Columns: {', '.join(df.columns)}")

Loaded labels.csv: 573 rows
Columns: text, label, note


### Check it before you train

This catches the problems that would otherwise show up as a confusing error
forty minutes from now — or, worse, as a model that trained on bad data and
told you nothing was wrong.

In [24]:
def check_dataset(df, labels):
    """
    Look for the things that break a training run or quietly ruin it.

    Returns (problems, warnings). Problems stop you; warnings are worth reading.
    """
    problems, warnings = [], []

    for column in ("text", "label"):
        if column not in df.columns:
            problems.append(f"No '{column}' column. Found: {', '.join(df.columns)}")
    if problems:
        return problems, warnings

    blank = df["text"].isna().sum() + (df["text"].astype(str).str.strip() == "").sum()
    if blank:
        problems.append(f"{blank} rows have an empty text column.")

    found = set(df["label"].dropna().astype(str).str.strip().unique())
    expected = set(labels)

    unknown = found - expected
    if unknown:
        problems.append(
            f"Labels in your CSV that aren't in LABELS: {sorted(unknown)}. "
            f"Check capitals and spaces — 'Hot_take' and 'hot_take' are different."
        )

    unused = expected - found
    if unused:
        warnings.append(f"Labels in LABELS that never appear in your CSV: {sorted(unused)}")

    counts = df["label"].value_counts()
    total = len(df)
    for label, count in counts.items():
        share = count / total
        if share > 0.70:
            problems.append(
                f"'{label}' is {share:.0%} of your data. The brief caps this at 70% — "
                f"above that the model learns to guess it. Collect more of the thin labels."
            )
        elif share > 0.60:
            warnings.append(f"'{label}' is {share:.0%} of your data. Getting lopsided.")

    if total < 150:
        warnings.append(
            f"{total} rows. The brief's floor for a submission is 150 and it asks "
            f"for 200. Training will still run — this is fine for the practice "
            f"data, and something to fix before you submit."
        )
    elif total < 200:
        warnings.append(f"{total} rows — under the 200 the brief asks for. Say so in your README.")

    duplicates = df["text"].duplicated().sum()
    if duplicates:
        warnings.append(
            f"{duplicates} duplicate posts. They'll land in different splits and "
            f"inflate your score — that's leakage."
        )

    for label, count in counts.items():
        if count < 20:
            warnings.append(
                f"Only {count} examples of '{label}'. Its test split will be tiny, "
                f"so its F1 will jump around between seeds."
            )

    return problems, warnings


problems, warnings = check_dataset(df, LABELS)

for w in warnings:
    print(f"[look at]  {w}")
for p in problems:
    print(f"[FIX ME]   {p}")

if not problems:
    print("\nData looks fine.")
    print()
    print(df["label"].value_counts().to_string())
else:
    print("\nFix the [FIX ME] lines before training.")


Data looks fine.

label
rule_question         214
risk_check            172
experience_request    126
miscellaneous          61


## 3. Split

70% train, 15% validation, 15% test.

The split is **stratified** — each split gets roughly the same label proportions
as the whole file. Without that, a small label can end up with nothing in the
test set and its F1 becomes undefined.

**Lock the test set and don't look at it again until evaluation.** Checking how
you're doing against the test set and then changing things is the slow-motion
version of training on it.

In [25]:
from sklearn.model_selection import train_test_split


def split_dataset(df, seed, test_size=0.15, val_size=0.15):
    """
    Stratified 70/15/15. Returns (train, val, test) DataFrames.

    Stratifying falls back to a plain random split if any label is too rare to
    stratify on — that's a warning sign about the dataset, not about the split.
    """
    def _stratify_on(frame):
        counts = frame["label"].value_counts()
        return frame["label"] if counts.min() >= 2 else None

    rest, test = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=_stratify_on(df)
    )
    val_share = val_size / (1 - test_size)
    train, val = train_test_split(
        rest, test_size=val_share, random_state=seed, stratify=_stratify_on(rest)
    )
    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True),
    )


def describe_splits(train, val, test, labels):
    """A table of per-label counts per split, and a note on the thin ones."""
    import pandas as pd

    table = pd.DataFrame({
        "train": train["label"].value_counts(),
        "val": val["label"].value_counts(),
        "test": test["label"].value_counts(),
    }).reindex(labels).fillna(0).astype(int)

    thin = [label for label in labels if table.loc[label, "test"] < 8]
    return table, thin


train_df, val_df, test_df = split_dataset(df, SEED)
table, thin = describe_splits(train_df, val_df, test_df, LABELS)

print(f"train {len(train_df)}  ·  val {len(val_df)}  ·  test {len(test_df)}\n")
print(table.to_string())

if thin:
    print()
    for label in thin:
        n = table.loc[label, "test"]
        print(f"[note] '{label}' has only {n} examples in the test split.")
    print()
    print("A metric computed on that few examples moves a lot between seeds.")
    print("Write that down now — next unit it's a diagnosis, not a surprise.")

train 401  ·  val 86  ·  test 86

                    train  val  test
label                               
rule_question         150   32    32
experience_request     88   19    19
risk_check            120   26    26
miscellaneous          43    9     9


## 4. Train

About ten minutes on a T4. It'll print a loss for each epoch as it goes.

**The loss curve tells you training is converging — not that the model learned
what you meant.** A model can drive loss down by memorising or by latching onto
something spurious, and neither is visible here. That's what next unit is for.

In [26]:
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed,
)
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support


def to_ids(df, labels):
    """Map label strings to the integer ids the model wants."""
    lookup = {label: i for i, label in enumerate(labels)}
    out = df.copy()
    out["labels"] = out["label"].astype(str).str.strip().map(lookup)
    return out[["text", "labels"]]


def compute_metrics(eval_pred):
    logits, gold = eval_pred
    predicted = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(gold, predicted),
        "f1_macro": f1_score(gold, predicted, average="macro", zero_division=0),
    }


def train_model(train_df, val_df, labels, seed, epochs, learning_rate, batch_size,
                base_model, quiet=False):
    """Fine-tune and return (trainer, tokenizer)."""
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model,
        num_labels=len(labels),
        id2label={i: l for i, l in enumerate(labels)},
        label2id={l: i for i, l in enumerate(labels)},
    )

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    train_ds = Dataset.from_pandas(to_ids(train_df, labels)).map(tokenize, batched=True)
    val_ds = Dataset.from_pandas(to_ids(val_df, labels)).map(tokenize, batched=True)

    args = TrainingArguments(
        output_dir="./takemeter_out",
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=10,
        seed=seed,
        report_to="none",
        disable_tqdm=quiet,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    return trainer, tokenizer


trainer, tokenizer = train_model(
    train_df, val_df, LABELS, SEED, EPOCHS, LEARNING_RATE, BATCH_SIZE, BASE_MODEL
)
print("\nTrained.")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 86/86 [00:00<00:00, 6083.21 examples/s]
/Users/smitchandi/Documents/Coding/ai201-p3_Model-Finetuning/ai201-p3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.303100,1.277161,0.383721,0.155271
2,1.196800,1.239511,0.441860,0.235053
3,1.128200,1.181303,0.406977,0.245671
4,0.978400,1.122268,0.523256,0.400135
5,0.823700,1.118368,0.511628,0.396356
6,0.731300,1.084669,0.558140,0.426050
7,0.586000,1.089730,0.569767,0.446270
8,0.507500,1.139940,0.604651,0.471812
9,0.476400,1.095851,0.569767,0.441828
10,0.414100,1.095452,0.569767,0.445076


/Users/smitchandi/Documents/Coding/ai201-p3_Model-Finetuning/ai201-p3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/smitchandi/Documents/Coding/ai201-p3_Model-Finetuning/ai201-p3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/smitchandi/Documents/Coding/ai201-p3_Model-Finetuning/ai201-p3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/smitchandi/Documents/Coding/ai201-p3_Model-Finetuning/ai201-p3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argum


Trained.


## 5. Results

Runs the model on the test split — the one it has never seen — and writes
`results.json`.

`results.json` records **how sure the model was**, whether its confident
guesses were right more often than its unsure ones. If one of your criteria is
about confidence, that's the number that tests it. It also records **the device
you trained on**, which you'll want next unit when your numbers and someone
else's don't match.

It also writes `test_split.csv`, the held-out posts themselves. Next unit
`baseline.py` reads that file so the baseline is scored on exactly these posts
and not a different sample. Two numbers from two different sets of posts aren't
a comparison.

**Commit both files.** Don't read the numbers closely yet — your criteria are
already filed, and reading results before you've written your standard is how
the standard quietly moves.

In [16]:
import json


def evaluate(trainer, tokenizer, test_df, labels, seed=None):
    """Score the model on the held-out split. Returns a plain dict."""
    from datasets import Dataset

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_ds = Dataset.from_pandas(to_ids(test_df, labels)).map(tokenize, batched=True)
    output = trainer.predict(test_ds)

    predicted = np.argmax(output.predictions, axis=-1)
    gold = np.array(output.label_ids)

    # How sure the model was about each guess. Softmax over the raw scores, then
    # the winning one. A criterion about confidence can't be tested without it.
    scores = output.predictions
    exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
    confidence = (exp / exp.sum(axis=-1, keepdims=True)).max(axis=-1)

    precision, recall, f1, support = precision_recall_fscore_support(
        gold, predicted, labels=list(range(len(labels))), zero_division=0
    )

    return {
        "seed": seed,
        "n_test": int(len(gold)),
        "accuracy": float(accuracy_score(gold, predicted)),
        "f1_macro": float(f1_score(gold, predicted, average="macro", zero_division=0)),
        "per_label": {
            label: {
                "precision": float(precision[i]),
                "recall": float(recall[i]),
                "f1": float(f1[i]),
                "support": int(support[i]),
            }
            for i, label in enumerate(labels)
        },
        "confusion": confusion_counts(gold, predicted, labels),
        "confidence": confidence_report(confidence, gold, predicted),
    }


def confidence_report(confidence, gold, predicted):
    """
    Are the model's confident guesses right more often than its unsure ones?

    If you wrote a criterion about confidence, this is what tests it. If the
    two thirds score about the same, the model's confidence means nothing —
    which is itself worth writing down.
    """
    right = gold == predicted
    ranked = np.argsort(confidence)
    third = max(1, len(confidence) // 3)
    least, most = ranked[:third], ranked[-third:]

    return {
        "mean_when_right": float(confidence[right].mean()) if right.any() else None,
        "mean_when_wrong": float(confidence[~right].mean()) if (~right).any() else None,
        "accuracy_most_confident_third": float(right[most].mean()),
        "accuracy_least_confident_third": float(right[least].mean()),
        "n_per_third": int(third),
    }


def confusion_counts(gold, predicted, labels):
    """Rows are true labels, columns are guesses. Plain nested dict."""
    matrix = {t: {p: 0 for p in labels} for t in labels}
    for g, p in zip(gold, predicted):
        matrix[labels[int(g)]][labels[int(p)]] += 1
    return matrix


results = evaluate(trainer, tokenizer, test_df, LABELS, seed=SEED)

# The device goes in the file, not just the console. Next unit you compare your
# numbers against your own other seeds and against other people's — and you
# cannot read a difference you can't attribute.
results["device"] = DEVICE
results["device_name"] = DEVICE_NAME
results["torch_version"] = torch.__version__
results["max_length"] = MAX_LENGTH

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

# The held-out posts themselves, for next unit's baseline to score.
test_df.to_csv("test_split.csv", index=False)

print(f"accuracy  {results['accuracy']:.3f}")
print(f"macro F1  {results['f1_macro']:.3f}\n")
for label, scores in results["per_label"].items():
    print(f"  {label:<16} F1 {scores['f1']:.3f}   (n={scores['support']})")

c = results["confidence"]
print(f"\nconfidence   most-sure third {c['accuracy_most_confident_third']:.3f}"
      f"   ·   least-sure third {c['accuracy_least_confident_third']:.3f}")

print()

print("Wrote results.json and test_split.csv. Commit them both — they're the")
print("evidence the run happened, and next unit reads test_split.csv.")


Map: 100%|██████████| 86/86 [00:00<00:00, 2478.53 examples/s]
/Users/smitchandi/Documents/Coding/ai201-p3_Model-Finetuning/ai201-p3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


accuracy  0.419
macro F1  0.230

  rule_question    F1 0.571   (n=32)
  experience_request F1 0.000   (n=19)
  risk_check       F1 0.348   (n=26)
  miscellaneous    F1 0.000   (n=9)

confidence   most-sure third 0.500   ·   least-sure third 0.321

Wrote results.json and test_split.csv. Commit them both — they're the
evidence the run happened, and next unit reads test_split.csv.


---
---

# ⬇️ Unit 6 starts here

Nothing below this line is needed in unit 5. Come back next unit.

---

## 6. Three seeds

Your score depends on which posts landed in the test split. Change the split,
change the score. So run it three times.

**This is one cell on purpose** — all three seeds run in a single execution, so
you get a comparable set or none at all. On a laptop CPU it's a few minutes per
seed; start it and go and do Milestone 3 while it runs.

If accuracy swings more than about ten points across the three, your dataset is
too small or too lopsided for a stable measure. That's a genuine diagnosis, not
a mistake.

**Set `RUN_LABEL` before each run.** `"before"` now; `"after"` when you come
back in Milestone 5. Leave it and the improvement overwrites the numbers you
are comparing it against.

In [27]:
SEEDS = [42, 7, 2024]

# WARNING: change this to "after" before you re-run this cell in Milestone 5.
#
# Without it the improvement run overwrites your before-numbers, and the file
# you committed as evidence quietly becomes the other set. Same idea as the
# --label flag on the eval scripts in units 2 and 4.
RUN_LABEL = "after"

all_results = []

for i, seed in enumerate(SEEDS, 1):
    print(f"\n{'=' * 60}\nSeed {seed}  ({i} of {len(SEEDS)})\n{'=' * 60}")

    tr, va, te = split_dataset(df, seed)
    t, tok = train_model(
        tr, va, LABELS, seed, EPOCHS, LEARNING_RATE, BATCH_SIZE, BASE_MODEL, quiet=True
    )
    r = evaluate(t, tok, te, LABELS, seed=seed)
    all_results.append(r)

    print(f"  accuracy {r['accuracy']:.3f}   macro F1 {r['f1_macro']:.3f}")

OUT = f"results_three_seeds_{RUN_LABEL}.json"

for r in all_results:
    r["device"] = DEVICE
    r["device_name"] = DEVICE_NAME
    r["torch_version"] = torch.__version__
    r["max_length"] = MAX_LENGTH

with open(OUT, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\n\n{'=' * 60}\nAll three done.\n{'=' * 60}")
print(f"Wrote {OUT}.")
if RUN_LABEL == "before":
    print('Re-running after your improvement? Set RUN_LABEL = \"after" first,')
    print('or you will overwrite the numbers you are comparing against.')

print(f"Wrote {OUT}. Commit it.")



Seed 42  (1 of 3)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 86/86 [00:00<00:00, 7465.95 examples/s]
/Users/smitchandi/Documents/Coding/ai201-p3_Model-Finetuning/ai201-p3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'loss': 1.3644, 'grad_norm': 1.4003775119781494, 'learning_rate': 1.930769230769231e-05, 'epoch': 0.38461538461538464}
{'loss': 1.3031, 'grad_norm': 2.48553729057312, 'learning_rate': 1.853846153846154e-05, 'epoch': 0.7692307692307693}
{'eval_loss': 1.2771605253219604, 'eval_accuracy': 0.38372093023255816, 'eval_f1_macro': 0.15527065527065528, 'eval_runtime': 0.8953, 'eval_samples_per_second': 96.056, 'eval_steps_per_second': 6.702, 'epoch': 1.0}


/Users/smitchandi/Documents/Coding/ai201-p3_Model-Finetuning/ai201-p3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'loss': 1.2743, 'grad_norm': 2.9079463481903076, 'learning_rate': 1.776923076923077e-05, 'epoch': 1.1538461538461537}
{'loss': 1.2639, 'grad_norm': 1.8803595304489136, 'learning_rate': 1.7e-05, 'epoch': 1.5384615384615383}
{'loss': 1.1968, 'grad_norm': 3.6047816276550293, 'learning_rate': 1.6230769230769233e-05, 'epoch': 1.9230769230769231}
{'eval_loss': 1.239511489868164, 'eval_accuracy': 0.4418604651162791, 'eval_f1_macro': 0.23505323505323505, 'eval_runtime': 0.9431, 'eval_samples_per_second': 91.192, 'eval_steps_per_second': 6.362, 'epoch': 2.0}


...

{'loss': 0.4448, 'grad_norm': 3.875878095626831, 'learning_rate': 3.923076923076923e-06, 'epoch': 8.076923076923077}
{'loss': 0.3714, 'grad_norm': 5.537056922912598, 'learning_rate': 3.153846153846154e-06, 'epoch': 8.461538461538462}
{'loss': 0.3864, 'grad_norm': 2.973099708557129, 'learning_rate': 2.384615384615385e-06, 'epoch': 8.846153846153847}
{'eval_loss': 1.169638991355896, 'eval_accuracy': 0.5232558139534884, 'eval_f1_macro': 0.49175579322638147, 'eval_runtime': 0.6835, 'eval_samples_per_second': 125.818, 'eval_steps_per_second': 8.778, 'epoch': 9.0}


/Users/smitchandi/Documents/Coding/ai201-p3_Model-Finetuning/ai201-p3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'loss': 0.3653, 'grad_norm': 5.976505279541016, 'learning_rate': 1.6153846153846157e-06, 'epoch': 9.23076923076923}
{'loss': 0.3061, 'grad_norm': 5.702436923980713, 'learning_rate': 8.461538461538463e-07, 'epoch': 9.615384615384615}
{'loss': 0.3511, 'grad_norm': 29.501108169555664, 'learning_rate': 7.692307692307694e-08, 'epoch': 10.0}
{'eval_loss': 1.1686387062072754, 'eval_accuracy': 0.5232558139534884, 'eval_f1_macro': 0.4923108796947806, 'eval_runtime': 0.5839, 'eval_samples_per_second': 147.296, 'eval_steps_per_second': 10.276, 'epoch': 10.0}
{'train_runtime': 131.3854, 'train_samples_per_second': 30.521, 'train_steps_per_second': 1.979, 'train_loss': 0.785210655285762, 'epoch': 10.0}


Map: 100%|██████████| 86/86 [00:00<00:00, 5083.65 examples/s]
/Users/smitchandi/Documents/Coding/ai201-p3_Model-Finetuning/ai201-p3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  accuracy 0.605   macro F1 0.480


All three done.
Wrote results_three_seeds_after.json.
Wrote results_three_seeds_after.json. Commit it.


### The numbers, as a table

Paste this into your README's run log. The Target and Verdict columns are yours
to fill in from `criteria.md` — the notebook doesn't know what you promised.

In [28]:
def spread_table(all_results, labels):
    """A markdown table of every measure across every seed."""
    seeds = [r["seed"] for r in all_results]
    header = f"| Measure | {' | '.join(f'Seed {s}' for s in seeds)} | Spread |"
    divider = "|---|" + "|".join(["---"] * (len(seeds) + 1)) + "|"
    rows = [header, divider]

    def row(name, values, fmt="{:.3f}"):
        spread = max(values) - min(values)
        cells = " | ".join(fmt.format(v) for v in values)
        return f"| {name} | {cells} | {fmt.format(spread)} |"

    rows.append(row("Overall accuracy", [r["accuracy"] for r in all_results]))
    rows.append(row("Macro F1", [r["f1_macro"] for r in all_results]))
    for label in labels:
        rows.append(row(f"F1 — `{label}`",
                        [r["per_label"][label]["f1"] for r in all_results]))
    return "\n".join(rows)


def stability_note(all_results):
    """Say plainly whether the spread makes these numbers trustworthy."""
    accuracies = [r["accuracy"] for r in all_results]
    spread = max(accuracies) - min(accuracies)
    if spread > 0.10:
        return (f"Accuracy moved {spread:.3f} across seeds — more than 10 points. "
                f"Your dataset is too small or too lopsided for a stable measure. "
                f"That IS a diagnosis. Write it down.")
    if spread > 0.05:
        return (f"Accuracy moved {spread:.3f} across seeds. Enough that a criterion "
                f"landing within that band can't really be called either way.")
    return f"Accuracy moved {spread:.3f} across seeds. Stable enough to trust."


print(spread_table(all_results, LABELS))
print()
print(stability_note(all_results))

| Measure | Seed 42 | Seed 7 | Seed 2024 | Spread |
|---|---|---|---|---|
| Overall accuracy | 0.593 | 0.581 | 0.605 | 0.023 |
| Macro F1 | 0.466 | 0.455 | 0.480 | 0.025 |
| F1 — `rule_question` | 0.657 | 0.636 | 0.677 | 0.041 |
| F1 — `experience_request` | 0.600 | 0.564 | 0.683 | 0.119 |
| F1 — `risk_check` | 0.607 | 0.621 | 0.561 | 0.059 |
| F1 — `miscellaneous` | 0.000 | 0.000 | 0.000 | 0.000 |

Accuracy moved 0.023 across seeds. Stable enough to trust.


## 7. Confusion matrix

Rows are true labels, columns are the model's guesses. The diagonal is right
answers; everything off it is a mistake with a direction.

**Your README needs this typed out as a markdown table.** The cell prints one
you can copy. An image of a matrix earns nothing.

Then find your biggest off-diagonal number and say what it means in one
sentence. Not "the model made mistakes" — *which* boundary it didn't learn, and
which way round.

In [29]:
def matrix_markdown(confusion, labels):
    """The confusion matrix as a markdown table, ready to paste."""
    header = f"| true \\ predicted | {' | '.join(labels)} |"
    divider = "|---|" + "|".join(["---"] * len(labels)) + "|"
    rows = [header, divider]
    for true_label in labels:
        cells = " | ".join(str(confusion[true_label][p]) for p in labels)
        rows.append(f"| **{true_label}** | {cells} |")
    return "\n".join(rows)


def biggest_confusion(confusion, labels):
    """The largest off-diagonal cell: (true, predicted, count)."""
    worst = None
    for true_label in labels:
        for predicted in labels:
            if true_label == predicted:
                continue
            count = confusion[true_label][predicted]
            if worst is None or count > worst[2]:
                worst = (true_label, predicted, count)
    return worst


confusion = all_results[0]["confusion"]

print(matrix_markdown(confusion, LABELS))
print()

true_label, predicted, count = biggest_confusion(confusion, LABELS)
reverse = confusion[predicted][true_label]

print(f"Biggest confusion: {count} real `{true_label}` posts called `{predicted}`.")
print(f"The other direction: {reverse} real `{predicted}` posts called `{true_label}`.")
print()
if count > reverse * 2 and count > 2:
    print(f"That's lopsided — the model leans toward `{predicted}` on this pair.")
    print(f"A direction like that usually points at your labelling rather than the")
    print(f"model. Your agreement report is the thing that can tell you which.")
else:
    print("Roughly symmetric — the model finds this pair hard in both directions,")
    print("which more often means the boundary itself is genuinely hard.")

| true \ predicted | rule_question | experience_request | risk_check | miscellaneous |
|---|---|---|---|---|
| **rule_question** | 22 | 5 | 5 | 0 |
| **experience_request** | 2 | 12 | 5 | 0 |
| **risk_check** | 7 | 2 | 17 | 0 |
| **miscellaneous** | 4 | 2 | 3 | 0 |

Biggest confusion: 7 real `risk_check` posts called `rule_question`.
The other direction: 5 real `rule_question` posts called `risk_check`.

Roughly symmetric — the model finds this pair hard in both directions,
which more often means the boundary itself is genuinely hard.


---

### Before you close this notebook

- [ ] `results_three_seeds.json` pushed (the last cell does it — check it said so)
- [ ] The spread table pasted into your README
- [ ] The confusion matrix **typed as a markdown table** in your README
- [ ] One sentence on your biggest off-diagonal number

Then go and do the agreement check against the staff set. It's the only
instrument you have that can tell a labelling problem from a model problem —
and Milestone 4 asks you to use it as exactly that.